In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

from statsmodels.stats.outliers_influence import variance_inflation_factor as VIF
from statsmodels.stats.anova import anova_lm

from ISLP import load_data
from ISLP.models import (
    ModelSpec as MS,
    summarize,
    poly
)

### Simple Linear Regression

In [ ]:
Boston = load_data("Boston")
Boston.columns

In [ ]:
# create design matrix with intercept and 'lstat' predictor only
X = pd.DataFrame({
    'intercept': np.ones(Boston.shape[0]),
    'lstat': Boston['lstat']
})
X[:4]

In [ ]:
y = Boston['medv']
model = sm.OLS(y, X)
results = model.fit()
summarize(results)

In [ ]:
results.summary()

In [ ]:
# get coefficient values
results.params

In [ ]:
# predict on new data
newX = pd.DataFrame({
    'intercept': [1, 1, 1],
    'lstat': [5, 10, 15]
})
pred = results.get_prediction(newX)
pred.predicted_mean

In [ ]:
# confidence intervals
pred.conf_int(alpha=0.05)

In [ ]:
# prediction intervals
pred.conf_int(obs=True, alpha=0.05)

In [ ]:
# for plotting a regression line on a scatter plot
def abline(ax, b, m, *args, **kwargs):
    "Add a line with slope m and intercept b to ax"
    xlim = ax.get_xlim()
    ylim = [m * xlim[0] + b, m * xlim[1] + b]
    ax.plot(xlim, ylim, *args, **kwargs)

ax = Boston.plot.scatter('lstat', 'medv')
abline(ax,
       results.params[0],
       results.params[1],
       'r--',
       linewidth=3)

In [ ]:
# residual vs y_hat, shows some evidence of non-linearity
_, ax = plt.subplots(figsize=(8,8))
ax.scatter(results.fittedvalues, results.resid)
ax.set_xlabel('Fitted value')
ax.set_ylabel('Residual')
ax.axhline(0, c='k', ls='--')

In [ ]:
# scatterplot with leverage statistic
infl = results.get_influence()
_, ax = plt.subplots(figsize=(8,8))
ax.scatter(np.arange(X.shape[0]), infl.hat_matrix_diag)
ax.set_xlabel('Index')
ax.set_ylabel('Leverage')
np.argmax(infl.hat_matrix_diag) # observation w largest leverage

### Multiple Linear Regression

In [ ]:
X = MS(['lstat', 'age']).fit_transform(Boston)
model1 = sm.OLS(y, X)
results1 = model1.fit()
summarize(results1)

In [ ]:
terms = Boston.columns.drop('medv')
X = MS(terms).fit_transform(Boston)
model = sm.OLS(y, X)
results = model.fit()
summarize(results)

In [ ]:
minus_age = Boston.columns.drop(['medv', 'age'])
Xma = MS(minus_age).fit_transform(Boston)
modelma = sm.OLS(y, Xma)
summarize(modelma.fit())

### Multivariate Goodness of Fit

In [ ]:
# compute VIF for all predictors (i.e. design matrix columns minus intercept)
vals = [VIF(X, i) for i in range(1, X.shape[1])]
vif = pd.DataFrame({
    'vif': vals,
}, index=X.columns[1:])
vif

### Interaction Terms

In [ ]:
X = MS(['lstat', 'age', ('lstat', 'age')]).fit_transform(Boston) # tell model builder to include interaction term between 'lstat' and 'age'
model2 = sm.OLS(y, X)
summarize(model2.fit())

### Non-linear Transformations of Predictors

In [ ]:
X = MS([poly('lstat', degree=2), 'age']).fit_transform(Boston)
model3 = sm.OLS(y, X)
results3 = model3.fit()
summarize(results3)

In [ ]:
anova_lm(results1, results3)
# F statistic shows that the polynomial model is statistically significantly better than the linear model

In [ ]:
# plotting residual against fitted value in the polynomial model shows a better fit, with less discernible pattern in the residuals
_, ax = plt.subplots(figsize=(8,8))
ax.scatter(results3.fittedvalues, results3.resid)
ax.set_xlabel('Fitted value')
ax.set_ylabel('Residual')
ax.axhline(0, c='k', ls='--')

### Qualitative Predictors

In [ ]:
Carseats = load_data('Carseats')
Carseats.columns
# we are interested in the ShelveLoc variable, which contains 'Bad', 'Medium', 'Good'. We will encode this with one-hot encoding later.

In [ ]:
allvars = list(Carseats.columns.drop('Sales'))
y = Carseats['Sales']
final = allvars + [('Income', 'Advertising'), ('Price', 'Age')] # add some interaction terms
X = MS(final).fit_transform(Carseats)
model = sm.OLS(y, X)
summarize(model.fit())
# ShelveLoc[Good] has a positive coefficient, showing that a good location has a positive association with sales
# ShelveLoc[Medium] has a smaller but  positive coefficient, showing that it's not as good as the Good location, but better than a Bad location.